# Solar filament segmentation: stage 2 at native resolution (5 folds)

Run on Kaggle with **GPU T4 x2** and **Internet on**, competition data attached. Prepares the data from
scratch (folds are deterministic, seed 2026, so they match every other stage).

Trains the stage-2 U-Net (ConvNeXt-tiny + ASPP, filament / centerline / boundary) on random 1024 px
crops of the 2048 px image (2 crops per image per epoch, 70% centred on a filament),
two folds at a time, then writes out-of-fold maps for all 707 images, the fold-0 standalone PQ, and
5-fold-average test maps under `maps/unet2048/`.

In [ ]:
# 1. Setup: code from the team repo, extra packages
import os, subprocess, sys, time
from pathlib import Path

T0 = time.time()
CODE = Path('/kaggle/working/filament-seg')
if not CODE.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'charan/fusion-native-unet',
                    'https://github.com/hsfcoulibaly/aml_project.git', '/tmp/aml_project'], check=True)
    subprocess.run(['cp', '-r', '/tmp/aml_project/Azumah', str(CODE)], check=True)
    print(subprocess.run(['git', '-C', '/tmp/aml_project', 'log', '-1', '--oneline'],
                         capture_output=True, text=True).stdout)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'segmentation-models-pytorch', 'timm',
                'pycocotools'], check=True)
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.chdir(CODE)
sys.path.insert(0, str(CODE))
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Stage 0 data: manifest, folds, labels, preprocessed images (~20 min)
WORK = Path('/kaggle/working/fseg')
WORK.mkdir(parents=True, exist_ok=True)
os.environ['FSEG_WORK_DIR'] = str(WORK)
if not (WORK / 'references.pkl').exists():
    !python scripts/prepare.py --modes consensus
import pandas as pd
print(pd.read_csv(WORK / 'manifest.csv').fold.value_counts().sort_index().to_dict())
print(f'elapsed {(time.time() - T0) / 3600:.2f} h')

In [ ]:
# 3. Train the 5 fold models, two at a time (one per GPU). Rerunning resumes from last_state.pt.
os.makedirs('logs', exist_ok=True)

def launch(fold, gpu):
    cmd = (f'CUDA_VISIBLE_DEVICES={gpu} python scripts/train_unet.py --fold {fold} --name unet2048_f{fold} '
           f'--size 2048 --crop 1024 --crops-per-image 2 --epochs 30 --batch 4 '
           f'--workers 2 --pq-every 2 --patience 8 --resume > logs/unet2048_f{fold}.log 2>&1')
    return subprocess.Popen(cmd, shell=True)

for pair in [(0, 1), (2, 3), (4,)]:
    procs = [launch(f, g) for g, f in enumerate(pair)]
    for p in procs:
        p.wait()
    for f in pair:
        print(f'fold {f}:', open(f'logs/unet2048_f{f}.log').read().strip().splitlines()[-1])
    print(f'elapsed {(time.time() - T0) / 3600:.2f} h', flush=True)

In [ ]:
# 4. Out-of-fold maps for all 707 images + standalone U-Net PQ on fold 0
!python scripts/predict_unet.py oof --name unet2048 --folds 0 1 2 3 4 --sweep-fold 0 --tta flip
print(f'elapsed {(time.time() - T0) / 3600:.2f} h')

In [ ]:
# 5. Test maps (5-fold average) + standalone U-Net submission
!python scripts/predict_unet.py test --name unet2048 --folds 0 1 2 3 4 --tta flip \
    --config /kaggle/working/fseg/unet/runs/unet2048_f0/pq_config.json --out /kaggle/working/submission_unet2048.csv
print(f'elapsed {(time.time() - T0) / 3600:.2f} h')

In [ ]:
# 6. Summary; drop the YOLO export (not needed downstream) to keep the output small
import shutil
shutil.rmtree(WORK / 'yolo', ignore_errors=True)
for f in range(5):
    h = pd.read_csv(WORK / 'unet' / 'runs' / f'unet2048_f{f}' / 'history.csv')
    b = h.loc[h.val_pq.idxmax()]
    print(f'fold {f}: {len(h)} epochs, best val PQ {b.val_pq:.4f} at epoch {int(b.epoch)}, '
          f'{h.epoch_s.mean() / 60:.1f} min/epoch')
print('whole-disk 1024 reference (BEST_PIPELINE.md): U-Net alone fold-0 OOF PQ 0.406')
print(pd.read_csv(WORK / 'unet' / 'runs' / 'unet2048_f0' / 'pq_sweep_fold0.csv').head(3).to_string())
print(f'total {(time.time() - T0) / 3600:.2f} h')